# 11 | 59-day dispatch replay and the RMSE failure

**Purpose.** Read the frozen holdout RMSE v1/v2 results together with the official dispatch replay (predicted score, realized score, capture), and see why the lowest point error need not produce the highest dispatch score.

**Inputs** (read only; this notebook does not train):

- `reports/holdout_rmse_v1/leaderboard.csv`, `reports/holdout_rmse_v2/leaderboard.csv`
- `reports/holdout_dispatch_replay/failure_analysis.json`
- `reports/holdout_rmse_v1/dispatch_daily.csv` and the v2 file of the same name
- `reports/holdout_rmse_v1/predictions.parquet`

**Checks:**

- All 35 recipes (16 from v1 and 19 from v2) appear, sorted by realized score.
- Four figures are written to `outputs/notebooks/figures/nb11_*.png`.
- `reports/phase_b/final_metrics.json` is not modified, and neither are the existing RMSE/MAE columns.
- The 59-day scores are not used to pick a new champion.

**Score.** The official score is `sum(price × power)`. It is not currency. `capture = realized / oracle`. The oracle is the same 3,321-window upper bound on the realized prices for every model.

The directory name `holdout` is historical. These 59 days were already used to rank RMSE. They are not an untouched holdout.

Stored outputs were removed from this public copy. The executed charts used Chinese labels and cannot be redrawn without the raw series. Numbers in the text are the recorded results.


## Read this first

- **The highest realized score is not the lowest RMSE.** The highest realized score is **v2 `transformer_contextual`** (daily mean **6214**, capture **0.690**). Its RMSE is 0.631, rank 12 in the v2 table.
- **v1 Transformer has the best point error (RMSE 0.514) and ranks 12th by score.** Daily mean realized score **5637**, almost on the weekday-slot mean (5644). Dropping the 30 December spike, the 58-day mean falls to **5196**.
- **On v1, the correlation of RMSE and realized score is +0.33.** Worse point error tends to come with a higher score. On v2 the correlation is about 0. RMSE on 96 points is not a proxy for the official score.
- **Mechanism: smoothing compresses peaks and troughs.** v1 Transformer predicts 66% of the true daily range, and the price-order Spearman correlation is 0.61. The dispatcher enumerates 8+8 windows on the forecast only, so the window can land in the wrong place. After more training, the same architecture returns to range ratio 0.93 and Spearman 0.71. RMSE gets worse and the score becomes the best in the field.
- **Do not pick another champion from these 59 days.** They were already used to rank RMSE. This notebook only diagnoses. The long write-up is `reports/holdout_dispatch_replay/failure_analysis.md`.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'phase_b').is_dir():
            return candidate
    raise FileNotFoundError('start this notebook from the repository root')


def show_figure(path: Path) -> None:
    display(Image(filename=str(path)))



REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES = REPO_ROOT / 'outputs' / 'notebooks' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
print({'repo_root': str(REPO_ROOT), 'figures': str(FIGURES.relative_to(REPO_ROOT))})


In [ ]:
V1 = REPO_ROOT / 'reports' / 'holdout_rmse_v1'
V2 = REPO_ROOT / 'reports' / 'holdout_rmse_v2'
FA = REPO_ROOT / 'reports' / 'holdout_dispatch_replay' / 'failure_analysis.json'
# Published copy is read-only. Frozen experiment outputs under reports/ are not
# distributed with this repository -- see DATA.md for how to regenerate them.
assert FA.exists(), (
    f'missing {FA}. This notebook reads frozen experiment outputs only. '
    'See DATA.md to obtain the data and regenerate reports/.'
)

v1 = pd.read_csv(V1 / 'leaderboard.csv')
v2 = pd.read_csv(V2 / 'leaderboard.csv')
v1['version'] = 'v1'
v2['version'] = 'v2'
board = pd.concat([v1, v2], ignore_index=True)
board['recipe'] = board['version'] + '/' + board['model']
board['rmse_rank'] = board.groupby('version')['RMSE'].rank(method='min').astype(int)
board['realized_rank'] = board['realized_profit_mean'].rank(method='min', ascending=False).astype(int)
board = board.sort_values('realized_profit_mean', ascending=False).reset_index(drop=True)
failure = json.loads(FA.read_text(encoding='utf-8'))
oracle = float(board['oracle_profit_mean'].iloc[0])
print(f'n_v1={len(v1)} n_v2={len(v2)} n_all={len(board)} oracle={oracle:.2f}')
print('v1 corr RMSE vs realized:', round(failure['corr_v1']['rmse_vs_realized'], 3))
print('v2 corr RMSE vs realized:', round(failure['corr_v2']['rmse_vs_realized'], 3))


## 1. Column definitions

| Column | Meaning |
| --- | --- |
| RMSE / MAE | Price error on 59 days, 5,664 quarter-hour points |
| `pred_profit_mean` | Predicted score: each day, `optimize_day` on the forecast, then the mean of `sum(forecast · power)` |
| `realized_profit_mean` | Realized score: the same locked power times the realized price `A` |
| `profit_gap` | Predicted minus realized. This is optimism, not window quality |
| `oracle_profit_mean` | Upper bound from enumerating windows on the realized price. It is the same for every model |
| `capture` | Realized score / oracle |

Dispatch is 59 separate loops: enumerate on the forecast, then settle on the realized price. It is not a search over windows using all 59 realized curves at once.


In [ ]:
show_cols = [
    'realized_rank', 'recipe', 'RMSE', 'rmse_rank', 'MAE',
    'pred_profit_mean', 'realized_profit_mean', 'profit_gap',
    'oracle_profit_mean', 'capture',
]
display_board = board[show_cols].copy()
display_board = display_board.rename(columns={
    'realized_rank': 'score_rank',
    'recipe': 'recipe',
    'rmse_rank': 'rmse_rank_in_panel',
    'pred_profit_mean': 'predicted_score',
    'realized_profit_mean': 'realized_score',
    'profit_gap': 'optimism_gap',
    'oracle_profit_mean': 'oracle',
    'capture': 'capture',
})
with pd.option_context('display.max_rows', 40, 'display.width', 160):
    display(
        display_board.style.format({
            'RMSE': '{:.4f}',
            'MAE': '{:.4f}',
            'predicted_score': '{:.1f}',
            'realized_score': '{:.1f}',
            'optimism_gap': '{:.1f}',
            'oracle': '{:.2f}',
            'capture': '{:.3f}',
        }).hide(axis='index')
    )


## 2. Realized-score bar chart

The dashed line is the oracle (9000.89). Every model is below it. The top few recipes differ by tens to about a hundred official score units. v1 Transformer nearly overlaps the weekday-slot mean.


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 10.5))
labels = board['recipe'].tolist()[::-1]
values = board['realized_profit_mean'].to_numpy()[::-1]
colors = ['#4C78A8' if item.startswith('v1/') else '#F58518' for item in labels]
ax.barh(labels, values, color=colors, height=0.72)
ax.axvline(oracle, color='0.35', linestyle='--', linewidth=1.2, label=f'oracle {oracle:.0f}')
ax.set_xlabel('59-day mean realized dispatch score')
ax.set_title('Realized dispatch score, all 35 recipes')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
p1 = FIGURES / 'nb11_fig1_realized_bar.png'
fig.savefig(p1, dpi=150, bbox_inches='tight')
plt.close(fig)
show_figure(p1)
print(p1.relative_to(REPO_ROOT))


## 3. RMSE against realized score

If RMSE were a proxy for the dispatch score, points would run from the upper left to the lower right. On v1 they tilt the other way: worse point error, slightly higher score. The four labeled names are the anchors used below.


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.6))
for version, color, marker in (('v1', '#4C78A8', 'o'), ('v2', '#F58518', 's')):
    slice_ = board.loc[board['version'] == version]
    ax.scatter(
        slice_['RMSE'],
        slice_['realized_profit_mean'],
        c=color,
        marker=marker,
        s=42,
        alpha=0.85,
        label=version,
        edgecolors='white',
        linewidths=0.4,
    )
annotate = {
    'v1/transformer_contextual': 'v1 Transformer\nRMSE#1',
    'v2/transformer_contextual': 'v2 Transformer\nscore rank 1',
    'v1/ridge_contextual_a1': 'v1 Ridge',
    'v1/lstm_contextual': 'v1 LSTM',
    'v1/climatology_weekday_slot': 'climatology',
    'v2/xgboost_contextual': 'v2 XGB\nRMSE#1',
}
for recipe, text in annotate.items():
    row = board.loc[board['recipe'] == recipe].iloc[0]
    ax.annotate(
        text,
        (row['RMSE'], row['realized_profit_mean']),
        textcoords='offset points',
        xytext=(6, 6),
        fontsize=8,
    )
ax.set_xlabel('Evaluation RMSE (lower is better)')
ax.set_ylabel('59-day mean realized score')
ax.set_title('Point error is not the official score')
ax.set_ylim(4400, 6450)
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
fig.tight_layout()
p2 = FIGURES / 'nb11_fig2_rmse_vs_realized.png'
fig.savefig(p2, dpi=150, bbox_inches='tight')
plt.close(fig)
show_figure(p2)
print(p2.relative_to(REPO_ROOT))


## 4. Why v1 Transformer is accurate on points and weak on score

`optimize_day` sees only the 96-point forecast, enumerates 3,321 legal 8+8 windows, and **locks** the power. RMSE averages all 96 points. The score depends on whether the cheapest 8 slots and the most expensive 8 slots are ordered correctly and far enough apart.

**Capture is not a second model.** The table's capture is mean realized score / mean oracle, which equals total realized score / total oracle. It is not the mean of daily capture. v1 Transformer's mean daily capture is 0.57 and the table shows 0.63, because 30 December almost hits the window and scores 31241.


In [ ]:
focus_order = [
    ('v1', 'transformer_contextual'),
    ('v2', 'transformer_contextual'),
    ('v1', 'ridge_contextual_a1'),
    ('v1', 'lstm_contextual'),
]
focus_map = {(row['version'], row['model']): row for row in failure['focus']}
rows = []
for version, name in focus_order:
    item = focus_map[(version, name)]
    rows.append({
        'recipe': f'{version}/{name}',
        'RMSE': item['RMSE'],
        'realized_score': item['realized_profit_mean'],
        'predicted_score': item['pred_profit_mean'],
        'optimism_gap': item['profit_gap'],
        'capture': item['capture'],
        'predicted_range_over_true': item['spread_ratio'],
        'Spearman': item['mean_spearman'],
        'mean_abs_charge_slots': item['mean_abs_charge_slots'],
        'days_capture_below_0_5': item['days_capture_below_0_5'],
        'negative_score_days': item['negative_days'],
    })
focus_frame = pd.DataFrame(rows)
display(
    focus_frame.style.format({
        'RMSE': '{:.3f}',
        'realized_score': '{:.1f}',
        'predicted_score': '{:.1f}',
        'optimism_gap': '{:.1f}',
        'capture': '{:.3f}',
        'predicted_range_over_true': '{:.2f}',
        'Spearman': '{:.2f}',
        'mean_abs_charge_slots': '{:.1f}',
    }).hide(axis='index')
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.2))
names = focus_frame['recipe'].str.replace('_contextual', '', regex=False).str.replace('_a1', '', regex=False)
x = np.arange(len(names))
axes[0].bar(x, focus_frame['predicted_range_over_true'], color=['#4C78A8', '#F58518', '#54A24B', '#E45756'])
axes[0].axhline(1.0, color='0.35', linestyle='--', linewidth=1)
axes[0].set_xticks(x, names, rotation=20, ha='right')
axes[0].set_ylabel('Predicted range / true range')
axes[0].set_title('Range: 1 means no compression')
axes[0].set_ylim(0, 1.15)
axes[0].grid(axis='y', alpha=0.3)
axes[1].bar(x, focus_frame['Spearman'], color=['#4C78A8', '#F58518', '#54A24B', '#E45756'])
axes[1].set_xticks(x, names, rotation=20, ha='right')
axes[1].set_ylabel('Spearman(realized price, forecast)')
axes[1].set_title('Ordering, which is what dispatch uses')
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)
fig.suptitle('v1 Transformer compresses the range and mis-orders high and low hours')
fig.tight_layout()
p3 = FIGURES / 'nb11_fig3_spread_spearman.png'
fig.savefig(p3, dpi=150, bbox_inches='tight')
plt.close(fig)
show_figure(p3)
print(p3.relative_to(REPO_ROOT))


Three mechanisms, on the same 59 forecast curves:

1. **Range compression.** v1 Transformer predicts 66% of the true daily range. The peak-to-trough gap that `optimize_day` sees is too small, so the predicted score is only 6627 (Ridge is 10503). The optimism gap is only 990: the model is honest about being small. The error is where the window sits.
2. **The 16 slots that determine the score are poorly ordered.** Spearman correlation 0.61, against 0.70+ for Ridge and v2 Transformer. RMSE on the oracle's 16 slots is 0.52, and RMSE on the other 80 slots is 0.41. Accuracy is spent on the quiet hours.
3. **The window is off by about five hours.** Zero exact `(tc, td)` hits in 59 days is not surprising across 3,321 windows. The charge start is off by 20 slots on average, about five hours.

v2 Transformer trains for more epochs after a 30-day stopping rule. RMSE worsens because the curve no longer sticks to the mean. The range ratio returns to 0.93 and Spearman correlation rises to 0.71, and the recipe becomes the highest realized score in the field. The same architecture and two training runs point in opposite directions. v1 LSTM is a sharper version of the same failure (range ratio 0.62, 8 negative-score days, capture 0.51). Ridge is the opposite case: the points are inaccurate, and the order of cheap and expensive hours is still usable.


## 5. Two days: 12 December misses the window, 30 December hits the spike

Shading is the charge block (blue) and discharge block (red) locked by v1 Transformer. The thin line is that day's realized `A`. The thick dashed line is its forecast.


In [ ]:
from src.phase_b.contracts import BLOCK_STEPS, STEPS_PER_DAY

pred = pd.read_parquet(V1 / 'predictions.parquet')
pred['times'] = pd.to_datetime(pred['times'])
pred['date'] = pred['times'].dt.normalize()
daily = pd.read_csv(V1 / 'dispatch_daily.csv')
daily['date'] = pd.to_datetime(daily['date']).dt.normalize()
tr_daily = daily.loc[daily['model'] == 'transformer_contextual'].set_index('date')

fig, axes = plt.subplots(2, 1, figsize=(9.4, 7.2), sharex=False)
cases = [
    (pd.Timestamp('2025-12-12'), '12 Dec, wrong window: realized -5343, oracle +9468'),
    (pd.Timestamp('2025-12-30'), '30 Dec, spike almost matched: realized +31241, oracle +31559'),
]
hours = np.arange(STEPS_PER_DAY) / 4.0
for ax, (day, title) in zip(axes, cases):
    day_frame = pred.loc[pred['date'] == day]
    actual = day_frame['A'].to_numpy()
    hat = day_frame['transformer_contextual'].to_numpy()
    row = tr_daily.loc[day]
    tc, td = int(row['tc']), int(row['td'])
    ax.plot(hours, actual, color='0.15', lw=1.6, label='realized A')
    ax.plot(hours, hat, color='#4C78A8', lw=1.8, ls='--', label='v1 Transformer forecast')
    ax.axvspan(tc / 4.0, (tc + BLOCK_STEPS) / 4.0, color='#4C78A8', alpha=0.18, label='locked charge, 8 slots')
    ax.axvspan(td / 4.0, (td + BLOCK_STEPS) / 4.0, color='#E45756', alpha=0.18, label='locked discharge, 8 slots')
    ax.set_title(title)
    ax.set_ylabel('price, source scale')
    ax.set_xlim(0, 24)
    ax.grid(alpha=0.3)
axes[0].legend(loc='upper left', fontsize=8, ncol=2)
axes[1].set_xlabel('hour')
fig.tight_layout()
p4 = FIGURES / 'nb11_fig4_two_days.png'
fig.savefig(p4, dpi=150, bbox_inches='tight')
plt.close(fig)
show_figure(p4)

without = tr_daily.drop(index=pd.Timestamp('2025-12-30'))['realized_profit'].mean()
print(
    f"v1 Transformer 59-day mean {tr_daily['realized_profit'].mean():.1f}; "
    f"58-day mean without 30 Dec {without:.1f}"
)
print(p4.relative_to(REPO_ROOT))


## 6. How to read the columns, and what not to do next

- A small RMSE only says the points are close. A high `pred_profit` only says enumeration on the model's own forecast looks valuable. **`realized_profit` is the replay score.**
- A large `profit_gap` means the forecast was too optimistic about peaks and troughs. A low `capture` means little of the perfect-information score was kept. Either one can be bad while the other is not.
- Do not retrain, reselect, or edit `optimize_day` using these 59 days. The useful next question is whether a head that keeps range and order beats a head trained only for 96-point RMSE on a slice that was not already used to rank RMSE.
